In [ ]:
pip install simpy

In [ ]:
"""
===========================================================
SIMULACIÓN DE UNA AGENCIA BANCARIA - DOS COLAS
(cola externa sin límite + cola interna con cupo máximo)

Modelo:
- Llegadas: Exponencial
- Atención: Lognormal
- Cola EXTERNA: sin límite de personas (esperan afuera / en la calle)
- Cola INTERNA: máximo 6 personas (esperan adentro, ya en la agencia)
- Los clientes pasan de la cola externa a la interna en cuanto se
  libera un cupo (cuando alguien de adentro empieza a ser atendido
  o termina y se va).
- A las 6:00 pm (hora de cierre): quien siga en la cola EXTERNA ya
  no puede entrar y se retira sin ser atendido. Quien ya esté en la
  cola INTERNA sigue siendo atendido hasta terminar, sin importar
  que la hora de cierre ya haya pasado.
===========================================================
"""
import simpy
import random
import math
from collections import defaultdict

# ==========================================================
# VARIABLES DE ENTRADA
# ==========================================================

HORA_INICIO = 9
HORA_CIERRE = 18
HORARIO_ATENCION = (HORA_CIERRE - HORA_INICIO) * 60   # 540 minutos

NUMERO_CAJEROS = 2

# Capacidad máxima de la cola/sala INTERNA (personas esperando adentro)
CAPACIDAD_COLA_INTERNA = 6

# --- Llegadas: EXPONENCIAL ---
TIEMPO_ENTRE_LLEGADAS = 2.88

# --- Atención: LOGNORMAL ---
TIEMPO_ATENCION_MEDIA = 5.848
TIEMPO_ATENCION_DESV  = 4.044

_var = TIEMPO_ATENCION_DESV ** 2
_sigma2 = math.log(1 + (_var / (TIEMPO_ATENCION_MEDIA ** 2)))
SIGMA_LOGNORMAL = math.sqrt(_sigma2)
MU_LOGNORMAL = math.log(TIEMPO_ATENCION_MEDIA) - (_sigma2 / 2)

# Generadores aleatorios independientes (llegadas / atención)
rng_llegadas = random.Random(50)
rng_atencion = random.Random(51)


# ==========================================================
# VARIABLES DE ESTADO
# ==========================================================

longitud_cola_externa = 0
longitud_cola_interna = 0

# ==========================================================
# VARIABLES DE SALIDA
# ==========================================================

numero_clientes_llegaron = 0
numero_clientes_atendidos = 0
numero_clientes_perdidos = 0     # se retiraron sin entrar (cierre)

lista_espera_externa = []        # tiempo esperando afuera hasta entrar
lista_espera_interna = []        # tiempo esperando adentro hasta ser atendido
lista_tiempo_atencion = []
lista_tiempo_sistema = []        # tiempo total desde que llegó hasta que salió

# ==========================================================
# VARIABLES AUXILIARES PARA INDICADORES (áreas bajo la curva)
# ==========================================================

area_cola_externa = 0.0
area_cola_interna = 0.0
ultimo_evento_externa = 0.0
ultimo_evento_interna = 0.0

tiempo_ocupado_total = 0.0


# ==========================================================
# CONVERSIÓN DE MINUTOS DE SIMULACIÓN A HORA REAL
# ==========================================================

def hora_real(minutos_simulacion):
    total_minutos = int(HORA_INICIO * 60 + minutos_simulacion)
    horas = total_minutos // 60
    minutos = total_minutos % 60
    return f"{horas:02d}:{minutos:02d}"


# ==========================================================
# ACTUALIZAR ÁREAS DE LAS COLAS (para promedio de longitud)
# ==========================================================

def actualizar_area_externa(env):
    global area_cola_externa, ultimo_evento_externa
    tiempo = env.now - ultimo_evento_externa
    area_cola_externa += longitud_cola_externa * tiempo
    ultimo_evento_externa = env.now

def actualizar_area_interna(env):
    global area_cola_interna, ultimo_evento_interna
    tiempo = env.now - ultimo_evento_interna
    area_cola_interna += longitud_cola_interna * tiempo
    ultimo_evento_interna = env.now


# ==========================================================
# PROCESO CLIENTE
# ==========================================================

def cliente(env, nombre, sala_interna, cajeros):

    global numero_clientes_atendidos
    global numero_clientes_perdidos
    global tiempo_ocupado_total
    global longitud_cola_externa
    global longitud_cola_interna

    llegada = env.now

    print(f"{hora_real(env.now)} -> {nombre} llega y entra a la cola EXTERNA")

    actualizar_area_externa(env)
    longitud_cola_externa += 1

    # Tiempo restante hasta el cierre oficial; si se agota antes de
    # conseguir un cupo interno, el cliente se retira sin ser atendido.
    tiempo_restante = max(0, HORARIO_ATENCION - env.now)

    with sala_interna.request() as solicitud_interna:

        resultado = yield solicitud_interna | env.timeout(tiempo_restante)

        actualizar_area_externa(env)
        longitud_cola_externa -= 1

        if solicitud_interna not in resultado:
            # Se cerró antes de conseguir un cupo interno: se va sin atención
            numero_clientes_perdidos += 1
            print(
                f"{hora_real(env.now)} -> {nombre} se RETIRA sin ser atendido "
                f"(cierre; seguía en la cola externa)"
            )
            return

        # ---- Consiguió un cupo en la cola/sala INTERNA ----
        entrada_interna = env.now
        espera_externa = entrada_interna - llegada
        lista_espera_externa.append(espera_externa)

        actualizar_area_interna(env)
        longitud_cola_interna += 1

        print(f"{hora_real(env.now)} -> {nombre} entra a la cola INTERNA")

        with cajeros.request() as solicitud_cajero:

            yield solicitud_cajero

            actualizar_area_interna(env)
            longitud_cola_interna -= 1

            inicio_atencion = env.now
            espera_interna = inicio_atencion - entrada_interna
            lista_espera_interna.append(espera_interna)

            print(f"{hora_real(env.now)} -> {nombre} inicia atención")

            servicio = rng_atencion.lognormvariate(MU_LOGNORMAL, SIGMA_LOGNORMAL)
            lista_tiempo_atencion.append(servicio)
            tiempo_ocupado_total += servicio

            yield env.timeout(servicio)

            numero_clientes_atendidos += 1
            lista_tiempo_sistema.append(env.now - llegada)

            print(f"{hora_real(env.now)} -> {nombre} sale")

        # Al salir de este 'with', se libera el cupo de la sala interna
        # automáticamente, permitiendo que otro cliente de la cola
        # externa pueda entrar (si todavía no ha cerrado).


# ==========================================================
# GENERADOR DE CLIENTES
# ==========================================================

def llegadas(env, sala_interna, cajeros):

    global numero_clientes_llegaron
    contador = 1

    while True:
        tiempo = rng_llegadas.expovariate(1 / TIEMPO_ENTRE_LLEGADAS)

        # No se generan más llegadas después de la hora de cierre
        if env.now + tiempo >= HORARIO_ATENCION:
            break

        yield env.timeout(tiempo)

        numero_clientes_llegaron += 1

        env.process(cliente(env, f"Cliente {contador}", sala_interna, cajeros))

        contador += 1


# ==========================================================
# EJECUCIÓN
# ==========================================================

env = simpy.Environment()

# Recurso que representa los 6 cupos físicos de la sala interna
sala_interna = simpy.Resource(env, capacity=CAPACIDAD_COLA_INTERNA)

# Recurso que representa los cajeros
cajeros = simpy.Resource(env, capacity=NUMERO_CAJEROS)

env.process(llegadas(env, sala_interna, cajeros))

# Sin límite fijo: termina cuando ya no hay más llegadas posibles
# Y todos los que entraron a la sala interna ya fueron atendidos.
env.run()

tiempo_total_operacion = env.now
minutos_extra = max(0, tiempo_total_operacion - HORARIO_ATENCION)


# ==========================================================
# INDICADORES DE DESEMPEÑO
# ==========================================================

print("\n")
print("="*65)
print("INDICADORES DE DESEMPEÑO")
print("="*65)

promedio_espera_externa = (
    sum(lista_espera_externa) / len(lista_espera_externa)
) if lista_espera_externa else 0

promedio_espera_interna = (
    sum(lista_espera_interna) / len(lista_espera_interna)
) if lista_espera_interna else 0

promedio_atencion = (
    sum(lista_tiempo_atencion) / len(lista_tiempo_atencion)
) if lista_tiempo_atencion else 0

promedio_sistema = (
    sum(lista_tiempo_sistema) / len(lista_tiempo_sistema)
) if lista_tiempo_sistema else 0

cola_externa_promedio = area_cola_externa / tiempo_total_operacion
cola_interna_promedio = area_cola_interna / tiempo_total_operacion

utilizacion = (
    tiempo_ocupado_total / (NUMERO_CAJEROS * tiempo_total_operacion)
) * 100

clientes_hora = numero_clientes_atendidos / (tiempo_total_operacion / 60)

print(f"Horario oficial                  : {HORA_INICIO}:00 - {HORA_CIERRE}:00")
print(f"Capacidad de la cola interna     : {CAPACIDAD_COLA_INTERNA} personas")
print(f"Clientes que llegaron            : {numero_clientes_llegaron}")
print(f"Clientes atendidos               : {numero_clientes_atendidos}")
print(f"Clientes perdidos (cierre)       : {numero_clientes_perdidos}")
print(f"Tiempo promedio espera EXTERNA   : {promedio_espera_externa:.2f} min")
print(f"Tiempo promedio espera INTERNA   : {promedio_espera_interna:.2f} min")
print(f"Tiempo promedio de atención      : {promedio_atencion:.2f} min")
print(f"Tiempo promedio en el sistema     : {promedio_sistema:.2f} min")
print(f"Longitud promedio cola EXTERNA   : {cola_externa_promedio:.2f} personas")
print(f"Longitud promedio cola INTERNA   : {cola_interna_promedio:.2f} personas "
      f"(máx. {CAPACIDAD_COLA_INTERNA})")
print(f"Utilización de cajeros           : {utilizacion:.2f}%")
print(f"Clientes atendidos por hora      : {clientes_hora:.2f}")
print(f"Hora real del último cliente     : {hora_real(tiempo_total_operacion)}")
print(f"Minutos extra tras el cierre     : {minutos_extra:.2f} min")

print("="*65)

09:01 -> Cliente 1 llega y entra a la cola EXTERNA
09:01 -> Cliente 1 entra a la cola INTERNA
09:01 -> Cliente 1 inicia atención
09:02 -> Cliente 2 llega y entra a la cola EXTERNA
09:02 -> Cliente 2 entra a la cola INTERNA
09:02 -> Cliente 2 inicia atención
09:04 -> Cliente 1 sale
09:05 -> Cliente 2 sale
09:05 -> Cliente 3 llega y entra a la cola EXTERNA
09:05 -> Cliente 3 entra a la cola INTERNA
09:05 -> Cliente 3 inicia atención
09:06 -> Cliente 4 llega y entra a la cola EXTERNA
09:06 -> Cliente 4 entra a la cola INTERNA
09:06 -> Cliente 4 inicia atención
09:08 -> Cliente 5 llega y entra a la cola EXTERNA
09:08 -> Cliente 5 entra a la cola INTERNA
09:12 -> Cliente 4 sale
09:12 -> Cliente 5 inicia atención
09:16 -> Cliente 3 sale
09:17 -> Cliente 5 sale
09:18 -> Cliente 6 llega y entra a la cola EXTERNA
09:18 -> Cliente 6 entra a la cola INTERNA
09:18 -> Cliente 6 inicia atención
09:18 -> Cliente 7 llega y entra a la cola EXTERNA
09:18 -> Cliente 7 entra a la cola INTERNA
09:18 -> Cli